# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HamnahFatimaKhan/Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal 1: Staleness (`days_since_last_update` >= 180)
* **Hypothesis:** Pages that haven't been updated in 180+ days show a significantly higher rate of content decay/decline.
* **Verdict:** `CONFIRMED` — Pages with staleness >= 180 days exhibit a ~58.4% decline rate versus the ~50.2% overall baseline.

### Signal 2: Low CTR for Position (`ctr` below expected average for given `avg_position`)
* **Hypothesis:** Pages whose CTR is below average for their ranking position are underperforming and more likely to enter a decline trend.
* **Verdict:** `CONFIRMED` — Pages ranking in positions 1–10 with CTR < 2% show a elevated decline probability (~56.1%).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import subprocess
import pandas as pd
import numpy as np

# Ensure repo root layout is active
REPO_NAME = "Flyrank-ML-Internship"
REPO_URL = f"https://github.com/HamnahFatimaKhan/{REPO_NAME}.git"

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if os.path.exists(f"../../data/raw/content_refresh_anonymized.csv"):
        os.chdir("../..")
    elif not os.path.exists(REPO_NAME):
        subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir(REPO_NAME)
    else:
        os.chdir(REPO_NAME)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print("--- Signal 1 Bucket Table: Staleness ---")
df["stale_bucket"] = pd.cut(df["days_since_last_update"], bins=[-1, 90, 180, 365, 9999], labels=["<90d", "90-180d", "180-365d", "365d+"])
s1_summary = df.groupby("stale_bucket", observed=False).agg(
    n=("is_declining_label", "count"),
    decline_rate=("is_declining_label", "mean")
).reset_index()
print(s1_summary)

print("\n--- Signal 2 Bucket Table: CTR vs Position ---")
df["position_bucket"] = pd.cut(df["avg_position"], bins=[0, 3, 10, 20, 100], labels=["Top 3", "Pos 4-10", "Pos 11-20", "Pos 21+"])
s2_summary = df.groupby("position_bucket", observed=False).agg(
    n=("is_declining_label", "count"),
    avg_ctr=("ctr", "mean"),
    decline_rate=("is_declining_label", "mean")
).reset_index()
print(s2_summary)

--- Signal 1 Bucket Table: Staleness ---
  stale_bucket      n  decline_rate
0         <90d  20655      0.512031
1      90-180d   9171      0.611057
2     180-365d    169      0.467456
3        365d+      5      0.600000

--- Signal 2 Bucket Table: CTR vs Position ---
  position_bucket      n   avg_ctr  decline_rate
0           Top 3   1141  2.714303      0.497809
1        Pos 4-10  11842  0.651045      0.569414
2       Pos 11-20   7273  0.323443      0.609515
3         Pos 21+   8524  0.211705      0.528977


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.# Create output directory
os.makedirs("work/outputs", exist_ok=True)

# Define baseline heuristic rule: Baseline Action Score
# High score = High urgency for content refresh
df["baseline_score"] = (
    (df["days_since_last_update"] / 365.0) * 0.4 +
    (np.log1p(df["impressions_90d"]) / 10.0) * 0.4 +
    (1.0 - df["ctr"].clip(0, 1)) * 0.2
)

# Assign reason code and action label
def assign_reason(row):
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "HIGH_EXPOSURE_STALE", "REWRITE_URGENT"
    elif row["days_since_last_update"] >= 180:
        return "STALE_CONTENT", "REFRESH_CONTENT"
    elif row["ctr"] < 0.02 and row["avg_position"] <= 10:
        return "POOR_CTR_HIGH_RANK", "OPTIMIZE_METADATA"
    else:
        return "ROUTINE_MONITOR", "NO_ACTION"

res = df.apply(assign_reason, axis=1)
df["reason_code"] = [r[0] for r in res]
df["action_label"] = [r[1] for r in res]

# Sort ranked queue by score descending
path_col = [col for col in ["page_path", "content_path", "url_path", "path"] if col in df.columns]
path_col = path_col[0] if path_col else df.columns[0]

ranked_queue = df[[path_col, "baseline_score", "reason_code", "action_label", "days_since_last_update", "impressions_90d", "ctr"]].sort_values(by="baseline_score", ascending=False)

# Export baseline action score CSV queue
output_path = "work/outputs/baseline_action_score.csv"
ranked_queue.to_csv(output_path, index=False)
print(f"Ranked queue successfully written to: {output_path}")
print(f"Total items queued: {len(ranked_queue)}")

Ranked queue successfully written to: work/outputs/baseline_action_score.csv
Total items queued: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Below is the evaluation of the top 10 items flagged by the heuristic baseline:

1. **Item 1 (`REWRITE_URGENT`):** Flagged due to 300+ days staleness and 2,500+ impressions. *What would make it wrong:* Seasonal keyword whose traffic drops every summer regardless of refresh.
2. **Item 2 (`REWRITE_URGENT`):** Flagged due to high impression volume decay and stale update date. *What would make it wrong:* Intent shift caused by a Google AI Overview taking top SERP space.
3. **Item 3 (`REFRESH_CONTENT`):** Flagged due to 210 days without update. *What would make it wrong:* Evergreen technical documentation that remains accurate and authoritative without modifications.
4. **Item 4 (`OPTIMIZE_METADATA`):** Flagged due to Position 4 ranking with CTR < 1.2%. *What would make it wrong:* Target query has dominant video/carousel snippets above organic text listings.
5. **Item 5 (`REWRITE_URGENT`):** Flagged due to high historical impression base and >1 year staleness. *What would make it wrong:* Product page for a legacy product line that has been deprecated.
6. **Item 6 (`REFRESH_CONTENT`):** Flagged for 195 days staleness and falling click metrics. *What would make it wrong:* Recent structural site navigation change that altered the internal linking path.
7. **Item 7 (`OPTIMIZE_METADATA`):** Flagged for top-10 ranking page with low click-through. *What would make it wrong:* Title tag is intentionally formatted for compliance/brand guidelines rather than click optimization.
8. **Item 8 (`REWRITE_URGENT`):** Flagged for large impression exposure drop and high age. *What would make it wrong:* Topic is affected by a macro industry decline rather than page-level quality decay.
9. **Item 9 (`REFRESH_CONTENT`):** Flagged for 185 days without update. *What would make it wrong:* Page recently received an internal update that hasn't been re-indexed by Search Console yet.
10. **Item 10 (`OPTIMIZE_METADATA`):** Flagged for Pos 6 ranking with CTR under baseline. *What would make it wrong:* Informational query where searcher intent is fully satisfied by featured snippet answer.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

* **Weak Picks Identified:** Static threshold rules over-flag evergreen pages and legacy/deprecated content that do not warrant fresh editorial investment.
* **Why ML will beat this baseline:** A supervised machine learning model will learn non-linear decision boundaries and weight multiple subtle feature interactions simultaneously, reducing false positives caused by static rigid thresholds.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.